# 09 - 帯域カットオフ：5500 cm⁻¹以上の選択（現ベスト LB 17.200）
**含水率予測コンペ スペクトル解析チャレンジ**

## このノートブックの目的
- **現在の最良提出（LB 17.200）を再現する**
- 手法: SNV（全波長）→ 5500 cm⁻¹以上を選択 → PLS(n=3) → clip[0.84, 298.58]
- GroupKFold / LOSO CV で評価し、提出CSVを生成

## 物理的根拠（5500 cm⁻¹カットオフ）
- 北海道大学 O-7論文：5200 cm⁻¹付近（O-H 伸縮+変角の結合音）が高含水率で**飽和**する
- 飽和帯域を含むと種間転移（calibration transfer）が悪化する
- 飽和帯域を除去（5500 cm⁻¹以上のみ使用）することで LB 19.59 → 17.20 に改善
- ※カットオフ値はCVで最適化したものではなく**物理知見に基づく選択**（CV最適化はLB悪化を招く）


## 0. ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('ライブラリ読み込み完了')


## 1. データの読み込み

In [ ]:
train = pd.read_csv('../data/train_near.csv', encoding='cp932')
test  = pd.read_csv('../data/test_near.csv',  encoding='cp932')

# species number・樹種は特徴量に絶対使わない（calibration transfer 制約）
META_COLS = ['sample number', 'species number', '樹種', '含水率']
SPEC_COLS = [c for c in train.columns if c not in META_COLS]

X_train = train[SPEC_COLS].values
X_test  = test[SPEC_COLS].values
y_train = train['含水率'].values
groups  = train['樹種'].values

# clip 範囲は学習データの含水率レンジ
y_min, y_max = y_train.min(), y_train.max()

print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')
print(f'含水率範囲 : {y_min:.2f} - {y_max:.2f} %')


## 2. SNV前処理（全波長で実施）
SNVは各スペクトルを正規化する処理。**帯域選択の前に全波長で実施する**のが重要
（部分波長だけで正規化すると平均・分散が変わり結果が変わる）。


In [ ]:
def snv(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / std

# 全1555波長でSNVを適用
X_train_snv = snv(X_train)
X_test_snv  = snv(X_test)

print('SNV前処理完了（全波長）:', X_train_snv.shape)


## 3. 5500 cm⁻¹以上の帯域を選択
SNV適用後に、波数 ≥ 5500 cm⁻¹ の列だけを抽出する。
5200 cm⁻¹付近の飽和帯域を除外することが狙い。


In [ ]:
CUTOFF = 5500.0  # cm⁻¹（物理知見に基づく固定値、CV最適化はしない）

# 列名は波数(cm⁻¹)。降順に並んでいる
wavenums = np.array([float(c) for c in SPEC_COLS])
band_mask = wavenums >= CUTOFF

X_train_band = X_train_snv[:, band_mask]
X_test_band  = X_test_snv[:, band_mask]

print(f'カットオフ: {CUTOFF} cm⁻¹以上')
print(f'選択波長数: {band_mask.sum()} / {len(SPEC_COLS)}')
print(f'X_train_band: {X_train_band.shape}')


## 4. GroupKFold CV（樹種グループ・リークなし評価）

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

N_COMPONENTS = 3   # 5500以上帯域での最適成分数（過学習せず最も転移する）
N_SPLITS     = 5

gkf = GroupKFold(n_splits=N_SPLITS)
oof_preds = np.zeros(len(y_train))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_band, y_train, groups)):
    pls = PLSRegression(n_components=N_COMPONENTS)
    pls.fit(X_train_band[tr_idx], y_train[tr_idx])
    pred = pls.predict(X_train_band[val_idx]).ravel()

    fold_rmse = rmse(y_train[val_idx], pred)
    print(f'Fold {fold+1}  val樹種: {set(groups[val_idx])}  RMSE: {fold_rmse:.4f}')
    oof_preds[val_idx] = pred

cv_rmse = rmse(y_train, oof_preds)
print(f'
GroupKFold CV RMSE : {cv_rmse:.4f}')
print('（CVはLBより約8〜9低めに出るため、参考値）')


## 5. LOSO CV（Leave One Species Out）
1樹種ずつ抜いて評価。テストが完全別樹種であることを模した最も実態に近い評価。
**LOSOメジアンがLBと最も相関する**（過去実証）。


In [ ]:
species_list = np.unique(groups)
loso_scores = {}

for sp in species_list:
    val_mask = groups == sp
    tr_mask  = ~val_mask
    pls = PLSRegression(n_components=N_COMPONENTS)
    pls.fit(X_train_band[tr_mask], y_train[tr_mask])
    loso_scores[sp] = rmse(y_train[val_mask], pls.predict(X_train_band[val_mask]).ravel())

loso_series = pd.Series(loso_scores).sort_values()
print('LOSO RMSE（樹種別）:')
print(loso_series.round(2).to_string())
print(f'
LOSO mean   : {loso_series.mean():.4f}')
print(f'LOSO median : {loso_series.median():.4f}  ← LBと相関する指標')


## 6. OOF予測の可視化

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_train, oof_preds, alpha=0.3, s=10)
ax.plot([y_min, y_max], [y_min, y_max], 'r--', linewidth=1)
ax.set_xlabel('実測含水率 [%]')
ax.set_ylabel('OOF予測値 [%]')
ax.set_title(f'GroupKFold OOF予測 vs 実測  (RMSE={cv_rmse:.2f})')
plt.tight_layout()
plt.show()


## 7. 最終モデル学習 & 提出CSV生成
全trainで学習し、testを予測。clip[0.84, 298.58]を適用して提出形式（ヘッダーなし2列）で保存。


In [ ]:
pls_final = PLSRegression(n_components=N_COMPONENTS)
pls_final.fit(X_train_band, y_train)

y_pred = pls_final.predict(X_test_band).ravel()
y_pred_clipped = np.clip(y_pred, y_min, y_max)

print(f'予測値（clip前）: min={y_pred.min():.2f}, max={y_pred.max():.2f}')
print(f'予測値（clip後）: min={y_pred_clipped.min():.2f}, max={y_pred_clipped.max():.2f}')

fname = '../submissions/submission_pls3_5500up.csv'
submission = pd.DataFrame({
    'sample number': test['sample number'],
    '含水率': y_pred_clipped
})
submission.to_csv(fname, index=False, header=False)
print(f'
提出CSV保存完了: {fname}  (LB 17.200 を再現)')
print(submission.head())
